# LAK-1 — Iceberg vs Delta vs Parquet (format comparison)

The foundation of the lakehouse track: the **same data** written three ways, and what each format
can and can't do — **ACID commits, time travel, schema evolution, MERGE/upsert**.

**Pre-requisite:** `make up` → Spark UI at http://localhost:4040. **Laptop-safe:** tiny data, all
under `.tmp/`; `make clean` recovers. Catalogs (see `conf/spark-defaults.conf`):
`iceberg_catalog` (Iceberg), `spark_catalog` (Delta), and a plain Parquet path.

In [ ]:
from common.spark_session import spark, display_df
from common.iceberg_meta import table_health, compare_health
from pyspark.sql import functions as F

ICE = "iceberg_catalog.default.lak1_iceberg"
DEL = "spark_catalog.default.lak1_delta"
PARQ = ".tmp/lak1_parquet"
spark

## Step 0 — the same data, three formats

In [ ]:
orders = (spark.range(1, 1001).withColumnRenamed("id", "order_id")
          .withColumn("customer_id", F.pmod(F.col("order_id"), F.lit(50)))
          .withColumn("amount", F.round(F.rand(seed=1) * 100, 2))
          .withColumn("status", F.lit("new")))

for t in (ICE, DEL):
    spark.sql(f"DROP TABLE IF EXISTS {t}")
orders.writeTo(ICE).using("iceberg").create()                 # Iceberg
orders.write.format("delta").mode("overwrite").saveAsTable(DEL)  # Delta
orders.write.mode("overwrite").parquet(PARQ)                   # plain Parquet (just files)

print("rows:",
      spark.table(ICE).count(), "iceberg /",
      spark.table(DEL).count(), "delta /",
      spark.read.parquet(PARQ).count(), "parquet")

## 1. ACID commits & snapshots

Iceberg and Delta record every commit as a **snapshot/version** (atomic, isolated). Parquet is
just files in a directory — an append is not atomic and has no version log.

In [ ]:
# append a second batch to each
more = (spark.range(1001, 1101).withColumnRenamed("id", "order_id")
        .withColumn("customer_id", F.pmod(F.col("order_id"), F.lit(50)))
        .withColumn("amount", F.round(F.rand(seed=2) * 100, 2))
        .withColumn("status", F.lit("new")))
more.writeTo(ICE).append()
more.write.format("delta").mode("append").saveAsTable(DEL)
more.write.mode("append").parquet(PARQ)

print("Iceberg snapshots:")
spark.sql(f"SELECT snapshot_id, operation FROM {ICE}.snapshots").show(truncate=False)
print("Delta history (version | operation):")
spark.sql(f"DESCRIBE HISTORY {DEL}").select("version", "operation").show(5, truncate=False)
print("Parquet: no version log — just", len(spark.read.parquet(PARQ).inputFiles()), "files on disk")

## 2. Time travel

Iceberg (`VERSION AS OF <snapshot_id>` / `TIMESTAMP AS OF`) and Delta (`VERSION AS OF <n>`) can
read an earlier version. Parquet cannot — the old state is gone once overwritten.

In [ ]:
first_snap = spark.sql(f"SELECT snapshot_id FROM {ICE}.snapshots ORDER BY committed_at").first()["snapshot_id"]
ice_v0 = spark.sql(f"SELECT COUNT(*) AS n FROM {ICE} VERSION AS OF {first_snap}").first()["n"]
del_v0 = spark.sql(f"SELECT COUNT(*) AS n FROM {DEL} VERSION AS OF 0").first()["n"]
print(f"Iceberg @ first snapshot: {ice_v0} rows   (now: {spark.table(ICE).count()})")
print(f"Delta   @ version 0     : {del_v0} rows   (now: {spark.table(DEL).count()})")
print("Parquet: no time travel — only the current files exist")

## 3. Schema evolution

Iceberg & Delta evolve the schema in metadata (`ALTER TABLE ... ADD COLUMN`), old data reads back
with `NULL`s. Plain Parquet has no table metadata — adding a column means a schema-mismatch you
must reconcile yourself (`mergeSchema`).

In [ ]:
for t in (ICE, DEL):
    spark.sql(f"ALTER TABLE {t} ADD COLUMN region STRING")
    print(f"{t.split('.')[-1]} columns now:", spark.table(t).columns)
print("old rows read back with NULL region (Iceberg):",
      spark.sql(f"SELECT region FROM {ICE} WHERE region IS NULL LIMIT 1").count() == 1)

## 4. MERGE / upsert

Iceberg & Delta support `MERGE INTO` (upserts, SCD, CDC sinks). Parquet has no MERGE — you'd
rewrite the whole dataset.

In [ ]:
updates = spark.createDataFrame([(1, 999.0), (5000, 1.0)], ["order_id", "amount"])
updates.createOrReplaceTempView("updates")
for t in (ICE, DEL):
    spark.sql(f"""
        MERGE INTO {t} d USING updates u ON d.order_id = u.order_id
        WHEN MATCHED THEN UPDATE SET d.amount = u.amount
        WHEN NOT MATCHED THEN INSERT (order_id, amount, status) VALUES (u.order_id, u.amount, 'merged')
    """)
print("order_id=1 amount after MERGE (iceberg):",
      spark.sql(f"SELECT amount FROM {ICE} WHERE order_id = 1").first()["amount"])
print("order_id=5000 inserted (delta):",
      spark.sql(f"SELECT status FROM {DEL} WHERE order_id = 5000").first()["status"])

## 5. Capability summary

| | Parquet | Delta | Iceberg |
|---|---|---|---|
| Atomic commits / snapshots | ❌ | ✅ | ✅ |
| Time travel | ❌ | ✅ (version) | ✅ (snapshot/timestamp) |
| Schema evolution (metadata) | ❌ | ✅ | ✅ |
| MERGE / upsert | ❌ | ✅ | ✅ |
| Hidden partitioning / partition evolution | ❌ | ❌ | ✅ |
| Engine-neutral spec | ✅ (raw) | mostly Spark | ✅ (multi-engine) |

The rest of this track digs into the **maintenance** the lakehouse formats need in production
(small files, snapshots, manifests, orphan files, MERGE cost) — that's where Iceberg/Delta earn
their keep *and* where they bite if you ignore upkeep.

In [ ]:
# Iceberg physical health (Delta: DESCRIBE DETAIL; Parquet: just a file count)
compare_health([table_health(spark, ICE, "iceberg (lak1)")])
det = spark.sql(f"DESCRIBE DETAIL {DEL}").select("numFiles", "sizeInBytes").first()
print(f"\nDelta:   {det['numFiles']} files, {det['sizeInBytes']:,} bytes")
print(f"Parquet: {len(spark.read.parquet(PARQ).inputFiles())} files (no metadata layer)")

## Teardown

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {ICE}")
spark.sql(f"DROP TABLE IF EXISTS {DEL}")
import shutil, os
shutil.rmtree(PARQ, ignore_errors=True)
print("Dropped tables + removed Parquet dir. (`make clean` clears all of .tmp.)")